In [1]:
from pathlib import Path
import numpy as np
import pandas as pd


from models import (PromptEncoder, TwoWayTransformer, SwinTransformer, MaskDecoder_Prompt)

/export/home/rstanciu/FM_thesis_Razvan/.venv/lib/python3.12/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


### Train/test split

In [2]:
csv_path="/mnt/data/spathak/CLaM-Annot-metadata.csv"

df=pd.read_csv(csv_path, delimiter=";")

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_columns", None)

to_drop=['MassShape','MassMargin',	'MassDensity',	'CalcificationMorphology',
         	'CalcificationDistribution',	'AsymmetryType',	'BIRADS', 'ConfidenceScore' ,
                'ImageHeight',	'ImageWidth' , 'StudyDate',	'CaseID',	'ViewID']

df=df.drop(to_drop, axis=1)



df = df.fillna("None")



In [3]:
df=df[(df['ROIPath'] != "None") & (df["AbnormalityType"] == "Mass")].reset_index()

In [5]:
df.to_pickle("/export/home/rstanciu/FM_thesis_Razvan/FM_thesis/df_masses.pkl")

In [10]:
from sklearn.model_selection import train_test_split

mass_df = df[df["AbnormalityType"] == "Mass"].copy()

patient_labels = (
    mass_df.groupby("PatientID")["PatientGroundtruth"]
    .agg(lambda x: "Malignant" if (x == "Malignant").any() else "Benign")
    .reset_index()
)

train_ids, test_ids = train_test_split(
    patient_labels["PatientID"],
    test_size=0.3,          # change this for different ratios
    stratify=patient_labels["PatientGroundtruth"],
    random_state=42,
)

train = df[df["PatientID"].isin(train_ids)].copy()
test = df[df["PatientID"].isin(test_ids)].copy()

print(train[(train['ROIPath'] != "None") & (train["AbnormalityType"] == "Mass")].shape)
print(test[(test['ROIPath'] != "None") & (test["AbnormalityType"] == "Mass")].shape)

(102, 12)
(57, 12)


In [11]:
train=train[(train['ROIPath'] != "None") & (train["AbnormalityType"] == "Mass")].reset_index()
test=test[(test['ROIPath'] != "None") & (test["AbnormalityType"] == "Mass")].reset_index()

In [12]:
train

,index,PatientID,ImageID,View,ROINum,BreastDensity,MammographicConspicuity,AbnormalityType,MammographyDiagnosis,PathologyDiagnosis,PatientGroundtruth,ImagePath,ROIPath
0,0,1,1.2.826.0.1.3680043.2.526.11.40.1608721769240233.24939309.435922,L CC,R1,D,Moderately visible,Mass,Malignant,Malignant,Malignant,CLaM-Annot/1.2.826.0.1.3680043.2.526.11.40.1608721769240233.24529438.829540/L CC_1.2.826.0.1.3680043.2.526.11.40.1608721769240233.24939312.782252/1.2.826.0.1.3680043.2.526.11.40.1608721769240233.24939309.435922_processed.png,CLaM-Annot/1.2.826.0.1.3680043.2.526.11.40.1608721769240233.24529438.829540/L CC_1.2.826.0.1.3680043.2.526.11.40.1608721769240233.24939312.782252/1.2.826.0.1.3680043.2.526.11.40.1608721769240233.24939309.435922_binarymask_R1.png
1,8,3,1.2.826.0.1.3680043.2.526.11.40.1608721769240233.3769087.9167098,L CC,R1,B,Clearly visible,Mass,Malignant,Malignant,Malignant,CLaM-Annot/1.2.826.0.1.3680043.2.526.11.40.1608721769240233.3769089.3298629/L CC_1.2.826.0.1.3680043.2.526.11.40.1608721769240233.3769090.2143573/1.2.826.0.1.3680043.2.526.11.40.1608721769240233.3769087.9167098_processed.png,CLaM-Annot/1.2.826.0.1.3680043.2.526.11.40.1608721769240233.3769089.3298629/L CC_1.2.826.0.1.3680043.2.526.11.40.1608721769240233.3769090.2143573/1.2.826.0.1.3680043.2.526.11.40.1608721769240233.3769087.9167098_binarymask_R1.png
2,10,3,1.2.826.0.1.3680043.2.526.11.40.1608721769240233.3940197.1505577,L MLO,R1,B,Clearly visible,Mass,Malignant,Malignant,Malignant,CLaM-Annot/1.2.826.0.1.3680043.2.526.11.40.1608721769240233.3769089.3298629/L MLO_1.2.826.0.1.3680043.2.526.11.40.1608721769240233.3940199.5591924/1.2.826.0.1.3680043.2.526.11.40.1608721769240233.3940197.1505577_processed.png,CLaM-Annot/1.2.826.0.1.3680043.2.526.11.40.1608721769240233.3769089.3298629/L MLO_1.2.826.0.1.3680043.2.526.11.40.1608721769240233.3940199.5591924/1.2.826.0.1.3680043.2.526.11.40.1608721769240233.3940197.1505577_binarymask_R1.png
3,26,6,1.2.826.0.1.3680043.2.526.11.40.16082154785849364.11278946.67427,L CC,R1,C,Clearly visible,Mass,Malignant,Malignant,Malignant,CLaM-Annot/1.2.826.0.1.3680043.2.526.11.40.16082154785849364.10880729.18033/L CC_1.2.826.0.1.3680043.2.526.11.40.16082154785849364.11278948.33240/1.2.826.0.1.3680043.2.526.11.40.16082154785849364.11278946.67427_processed.png,CLaM-Annot/1.2.826.0.1.3680043.2.526.11.40.16082154785849364.10880729.18033/L CC_1.2.826.0.1.3680043.2.526.11.40.16082154785849364.11278948.33240/1.2.826.0.1.3680043.2.526.11.40.16082154785849364.11278946.67427_binarymask_R1.png
4,27,6,1.2.826.0.1.3680043.2.526.11.40.16082154785849364.11043997.85220,L MLO,R1,B,Moderately visible,Mass,Benign,Malignant,Malignant,CLaM-Annot/1.2.826.0.1.3680043.2.526.11.40.16082154785849364.10880729.18033/L MLO_1.2.826.0.1.3680043.2.526.11.40.16082154785849364.11043999.19768/1.2.826.0.1.3680043.2.526.11.40.16082154785849364.11043997.85220_processed.png,CLaM-Annot/1.2.826.0.1.3680043.2.526.11.40.16082154785849364.10880729.18033/L MLO_1.2.826.0.1.3680043.2.526.11.40.16082154785849364.11043999.19768/1.2.826.0.1.3680043.2.526.11.40.16082154785849364.11043997.85220_binarymask_R1.png
...,...,...,...,...,...,...,...,...,...,...,...,...,...
97,481,98,1.2.826.0.1.3680043.2.526.11.40.1608721769240233.19602169.379600,R CC,R1,C,Clearly visible,Mass,Malignant,Malignant,Malignant,CLaM-Annot/1.2.826.0.1.3680043.2.526.11.40.1608721769240233.19529650.751250/R CC_1.2.826.0.1.3680043.2.526.11.40.1608721769240233.19602172.805698/1.2.826.0.1.3680043.2.526.11.40.1608721769240233.19602169.379600_processed.png,CLaM-Annot/1.2.826.0.1.3680043.2.526.11.40.1608721769240233.19529650.751250/R CC_1.2.826.0.1.3680043.2.526.11.40.1608721769240233.19602172.805698/1.2.826.0.1.3680043.2.526.11.40.1608721769240233.19602169.379600_binarymask_R1.png
98,482,98,1.2.826.0.1.3680043.2.526.11.40.1608721769240233.19886727.569973,R MLO,R1,D,Clearly visible,Mass,Malignant,Malignant,Malignant,CLaM-Annot/1.2.826.0.1.3680043.2.526.11.40.1608721769240233.19529650.751250/R MLO_1.2.826.0.1.3680043.2.526

In [13]:
train.to_pickle("train_ZGT.pkl")
test.to_pickle("test_ZGT.pkl")


In [14]:
def check(df_subset):
    return (
        df_subset.groupby("PatientID")["PatientGroundtruth"]
        .first()
        .value_counts()
    )

print(check(train))
print(check(test))


PatientGroundtruth
Malignant    29
Benign       17
Name: count, dtype: int64
PatientGroundtruth
Malignant    13
Benign        8
Name: count, dtype: int64
